# Lyapunov-PPO Training — Colab

**Runtime:** GPU (T4 or A100)  
**Target:** 2048 envs, ~50–200k SPS, checkpoints to Google Drive

Make sure **Runtime → Change runtime type → T4 GPU** is selected before running.

## 1. Mount Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/lyapunov-rl'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
print(f'Drive mounted. Working dir: {DRIVE_DIR}')

In [ ]:
# Option A: clone from GitHub (if repo is public/private)
# !git clone https://github.com/YOUR_USERNAME/lyapunov-rl.git /content/lyapunov-rl

# Option B: copy from Google Drive (upload a zip first)
# !cp /content/drive/MyDrive/lyapunov-rl/lyapunov-rl.zip /content/
# !unzip -q /content/lyapunov-rl.zip -d /content/

# Option C: copy individual folders from Drive (if synced via Drive desktop)
import shutil, os
SRC = '/content/drive/MyDrive/lyapunov-rl/code'  # adjust to where you put the code
DST = '/content/lyapunov-rl'
if os.path.exists(SRC):
    shutil.copytree(SRC, DST, dirs_exist_ok=True)
    print(f'Copied from Drive: {SRC} -> {DST}')
else:
    print(f'Source not found: {SRC}')
    print('Uncomment one of the options above to get the code onto Colab.')

## 2. Install Dependencies

In [ ]:
# JAX GPU version is already installed on Colab — upgrade to match local version
!pip install -q --upgrade \
    'jax[cuda12]' \
    equinox==0.11.10 \
    optax==0.2.4 \
    orbax-checkpoint==0.6.4 \
    wandb==0.25.1

# rebound not needed on Colab (validation only, runs on Mac)
print('Dependencies installed.')

In [ ]:
import jax
print(f'JAX version: {jax.__version__}')
print(f'Devices: {jax.devices()}')
print(f'Default backend: {jax.default_backend()}')
# Should show: [CudaDevice(id=0)] and 'gpu'

## 3. Setup Path & W&B

In [ ]:
import sys
sys.path.insert(0, '/content/lyapunov-rl')

# Quick import check
from shared.constants import EPSILON_MARS
from shared.obs import obs_dim
from train.env.trajectory_env import EnvParams
from train.agent.train import train
from train.agent.ppo import PPOHyperParams
print(f'Imports OK. EPSILON_MARS = {EPSILON_MARS:.3e}')

In [ ]:
import wandb
wandb.login()  # will prompt for API key — paste from wandb.ai/settings

## 4. Warmup Test (32 envs, 10 updates)

In [ ]:
import time
import jax
import jax.numpy as jnp
from train.env.trajectory_env import reset, step, EnvParams
from train.agent.ppo import PPOHyperParams

# Quick SPS benchmark before full training
params = EnvParams()
keys = jax.random.split(jax.random.PRNGKey(0), 2048)
batch_reset = jax.jit(jax.vmap(reset, in_axes=(0, None)), static_argnums=(1,))
batch_step  = jax.jit(jax.vmap(step,  in_axes=(0, 0, None)), static_argnums=(2,))

states, obss = batch_reset(keys, params)
actions = jnp.zeros((2048, 3))

# Warmup JIT
states, obss, _, _, _ = batch_step(states, actions, params)
jax.block_until_ready(states.sc_pos)

# Benchmark
t0 = time.time()
for _ in range(100):
    states, obss, _, _, _ = batch_step(states, actions, params)
jax.block_until_ready(states.sc_pos)
elapsed = time.time() - t0

sps = 100 * 2048 / elapsed
print(f'2048 envs: {sps:,.0f} steps/sec  ({elapsed:.2f}s for 100 steps)')
print(f'Expected: 50k–200k SPS on T4, 150k–500k on A100')

## 5. Training — Lyapunov PPO

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
N_ENVS        = 2048
N_STEPS       = 128      # rollout length per update
TOTAL_UPDATES = 5000     # ~1.3B frames
SEED          = 0
CHECKPOINT_DIR = f'{DRIVE_DIR}/checkpoints/lyapunov_seed{SEED}'

hp = PPOHyperParams(
    lambda_lyap = 0.1,
    lr          = 3e-4,
    clip_eps    = 0.2,
    c_vf        = 1.0,
    c_ent       = 0.001,
    n_epochs    = 3,
    n_minibatches = 8,    # larger minibatches for GPU efficiency
)

env_params = EnvParams()

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoints -> {CHECKPOINT_DIR}')
print(f'Total frames: {TOTAL_UPDATES * N_ENVS * N_STEPS:,}')

In [ ]:
from train.agent.train import train

train_state = train(
    n_envs        = N_ENVS,
    n_steps       = N_STEPS,
    total_updates = TOTAL_UPDATES,
    seed          = SEED,
    use_wandb     = True,
    checkpoint_dir= CHECKPOINT_DIR,
    hp            = hp,
    env_params    = env_params,
)

## 6. Training — Unconstrained Baseline

In [ ]:
CHECKPOINT_DIR_BASE = f'{DRIVE_DIR}/checkpoints/unconstrained_seed{SEED}'
os.makedirs(CHECKPOINT_DIR_BASE, exist_ok=True)

hp_base = PPOHyperParams(
    lambda_lyap = 0.0,   # <-- only difference
    lr          = 3e-4,
    clip_eps    = 0.2,
    c_vf        = 1.0,
    c_ent       = 0.001,
    n_epochs    = 3,
    n_minibatches = 8,
)

train_state_base = train(
    n_envs        = N_ENVS,
    n_steps       = N_STEPS,
    total_updates = TOTAL_UPDATES,
    seed          = SEED,
    use_wandb     = True,
    checkpoint_dir= CHECKPOINT_DIR_BASE,
    hp            = hp_base,
    env_params    = env_params,
)

## 7. Quick Eval (energy error over time)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from train.agent.networks import sample_action
from shared.constants import EPSILON_MARS, MU_SUN

def rollout_energy_curve(ts, env_p, seed=0, n_steps=9600):
    """Roll out policy for a full episode, return energy error over time."""
    key = jax.random.PRNGKey(seed)
    state, obs = reset(key, env_p)
    errors = []
    for i in range(n_steps):
        action, _ = sample_action(ts.policy, obs, jax.random.PRNGKey(i))
        state, obs, _, done, info = step(state, action, env_p)
        errors.append(float(info['energy_error']))
        if done:
            break
    return np.array(errors)

print('Rolling out Lyapunov policy...')
errors_lyap = rollout_energy_curve(train_state, env_params)
print('Rolling out baseline policy...')
errors_base = rollout_energy_curve(train_state_base, env_params)

times_days = np.arange(len(errors_lyap)) * env_params.dt / 86400

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(times_days[:len(errors_lyap)], errors_lyap, label='Lyapunov PPO', color='#2ecc71')
ax.plot(times_days[:len(errors_base)], errors_base[:len(errors_lyap)],
        label='Unconstrained PPO', color='#e74c3c')
ax.set_xlabel('Time (days)')
ax.set_ylabel('|ε - ε_target| / NORM_ENERGY')
ax.set_title('Energy Error vs Time')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{DRIVE_DIR}/energy_curve.png', dpi=150)
plt.show()
print(f'Saved to {DRIVE_DIR}/energy_curve.png')

## 8. Resume from Checkpoint

In [ ]:
# To resume a training run after Colab disconnects:
import equinox as eqx
import jax
from train.agent.networks import create_networks
from train.agent.ppo import create_train_state, PPOHyperParams
from shared.obs import obs_dim
from shared.constants import EPSILON_MARS

def load_checkpoint(prefix, obs_dim_val, epsilon_target=float(EPSILON_MARS)):
    """Load policy/value/lyapunov from checkpoint files."""
    key = jax.random.PRNGKey(0)
    policy, value_net, lyap_net = create_networks(
        obs_dim_val, epsilon_target=epsilon_target, key=key
    )
    policy    = eqx.tree_deserialise_leaves(f'{prefix}_policy.eqx', policy)
    value_net = eqx.tree_deserialise_leaves(f'{prefix}_value.eqx', value_net)
    lyap_net  = eqx.tree_deserialise_leaves(f'{prefix}_lyapunov.eqx', lyap_net)
    return policy, value_net, lyap_net

# Example:
# prefix = f'{DRIVE_DIR}/checkpoints/lyapunov_seed0/step_1000'
# policy, value_net, lyap_net = load_checkpoint(prefix, obs_dim(3))
# print('Checkpoint loaded.')